# Clean WP Comments: Display + Training

Creates two outputs:
1. `singapore_wp_comments_display.csv` for website display
2. `singapore_wp_comments_training.csv` for model training (labelling now that the project is done)


In [20]:
import re
import html
import unicodedata
import pandas as pd

INPUT_JSON = 'singapore_wp_comments.json'
DISPLAY_OUT = 'singapore_wp_comments_display.csv'
TRAIN_OUT = 'singapore_wp_comments_training.csv'
MIN_WORDS = 5




print('Input:', INPUT_JSON)
print('Display output:', DISPLAY_OUT)
print('Training output:', TRAIN_OUT)
print('Minimum words (URL-excluded):', MIN_WORDS)


Input: singapore_wp_comments.json
Display output: singapore_wp_comments_display.csv
Training output: singapore_wp_comments_training.csv
Minimum words (URL-excluded): 5


In [21]:
df = pd.read_json(INPUT_JSON)
print('Loaded rows:', len(df))
print('Columns:', len(df.columns))

# Convert epoch-based time columns to SGT strings
time_cols = [c for c in df.columns if c.endswith('_utc') or c in ['created', 'edited']]
for col in time_cols:
    numeric_ts = pd.to_numeric(df[col], errors='coerce')
    sgt = pd.to_datetime(numeric_ts, unit='s', utc=True, errors='coerce').dt.tz_convert('Asia/Singapore')
    df[f'{col}_sgt'] = sgt.dt.strftime('%Y-%m-%d %H:%M:%S %z')

print('Timing columns converted to SGT:', time_cols)


Loaded rows: 13538
Columns: 81
Timing columns converted to SGT: ['approved_at_utc', 'banned_at_utc', 'created', 'created_utc', 'edited', 'retrieved_utc', 'updated_utc', 'author_created_utc']


In [22]:
def repair_mojibake(text: str) -> str:
    # Repair common UTF-8 -> Latin-1/CP1252 mojibake such as '???'.
    markers = ('??', '???', '???', '?', '?')
    if any(m in text for m in markers):
        try:
            fixed = text.encode('latin1').decode('utf-8')
            if fixed.count('\uFFFD') <= text.count('\uFFFD'):
                return fixed
        except UnicodeError:
            pass
    return text

def normalize_text(text: str) -> str:
    text = '' if text is None else str(text)
    text = html.unescape(text)
    text = repair_mojibake(text)
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[\u200B-\u200D\uFEFF]', '', text)
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', ' ', text)
    # Normalize all common line separators to spaces
    text = re.sub(r'\r\n|[\r\n\u2028\u2029]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_markdown_urls(text: str) -> str:
    text = re.sub(r'\[([^\]]+)\]\((https?://[^)]+)\)', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'https?://\S+', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def word_count(text: str) -> int:
    return len(re.findall(r'\b\w+\b', text))

def alpha_ratio(text: str) -> float:
    chars = [c for c in text if not c.isspace()]
    if not chars:
        return 0.0
    alpha = sum(c.isalpha() for c in chars)
    return alpha / len(chars)

def url_count(text: str) -> int:
    return len(re.findall(r'https?://\S+', text, flags=re.IGNORECASE))


In [23]:
work = df.copy()
work['author_norm'] = work['author'].fillna('').astype(str).str.lower().str.strip()
work['body_raw'] = work['body'].fillna('').astype(str)
work['body_norm'] = work['body_raw'].map(normalize_text)
work['body_nourl'] = work['body_norm'].map(remove_markdown_urls)
work['word_count'] = work['body_nourl'].map(word_count)
work['alpha_ratio'] = work['body_nourl'].map(alpha_ratio)
work['url_count'] = work['body_norm'].map(url_count)

invalid_bodies = {'', '[deleted]', '[removed]'}
work['is_invalid_body'] = work['body_norm'].str.lower().isin(invalid_bodies)

print('Prepared engineered text features.')
print('replacement_char count:', int(work['body_norm'].str.contains(r'\uFFFD', regex=True, na=False).sum()))
print('zero_width count:', int(work['body_raw'].str.contains(r'[\u200B-\u200D\uFEFF]', regex=True, na=False).sum()))
mojibake_markers = [
    '\u00E2\u20AC\u2122',  # ???
    '\u00E2\u20AC\u0153',  # ???
    '\u00E2\u20AC',          # ??
    '\u00C3',                  # ?
    '\u00C2'                   # ?
]
remaining_mojibake_markers = sum(
    int(work['body_norm'].str.contains(m, regex=False, na=False).sum())
    for m in mojibake_markers
)
print('remaining_mojibake_markers:', remaining_mojibake_markers)
print('has_wp_training count:', int(work['body_nourl'].str.contains(r'\bwp\b', case=False, regex=True, na=False).sum()))


Prepared engineered text features.
replacement_char count: 0
zero_width count: 18
remaining_mojibake_markers: 0
has_wp_training count: 10890


In [24]:
# Shared base for both display and training (ensures same row set)
base_df = work.copy()
base_df = base_df[~base_df['is_invalid_body']]
base_df = base_df[base_df['word_count'] >= MIN_WORDS]
base_df = base_df.drop_duplicates(subset=['body_norm'])

# Display dataset (URLs kept)
display_df = base_df.copy()
display_df['body_display'] = display_df['body_norm']

display_cols = [
    'author', 'body_display', 'name', 'permalink', 'parent_id', 'score', 'subreddit_name_prefixed',
    'created_utc_sgt', 'word_count'
]
display_cols = [c for c in display_cols if c in display_df.columns]
display_df[display_cols].to_csv(DISPLAY_OUT, index=False, encoding='utf-8-sig')

print('DISPLAY rows:', len(display_df))
print('DISPLAY total words:', int(display_df['word_count'].sum()))


DISPLAY rows: 13214
DISPLAY total words: 1138740


In [25]:
# Training dataset (same rows, URLs removed only in text field)
train_df = base_df.copy()
train_df['body_training'] = train_df['body_nourl']

train_cols = [
    'author', 'body_training', 'name', 'permalink', 'parent_id', 'score', 'subreddit_name_prefixed',
    'created_utc_sgt', 'word_count', 'alpha_ratio', 'url_count'
]
train_cols = [c for c in train_cols if c in train_df.columns]
train_df[train_cols].to_csv(TRAIN_OUT, index=False, encoding='utf-8-sig')

print('TRAIN rows:', len(train_df))
print('TRAIN total words:', int(train_df['word_count'].sum()))


TRAIN rows: 13214
TRAIN total words: 1138740


In [27]:
# Corpus-level stats for both outputs
def corpus_stats(df, text_col, name):
    text_series = df[text_col].fillna('').astype(str)
    tokens_per_row = text_series.str.findall(r'\b\w+\b')
    total_words = int(tokens_per_row.str.len().sum())
    unique_words = len({tok.lower() for row in tokens_per_row for tok in row})
    return {
        'dataset': name,
        'Number of records': int(len(df)),
        'Total words': total_words,
        'Unique words (types)': int(unique_words),
    }

stats_df = pd.DataFrame([
    corpus_stats(display_df, 'body_display', 'display'),
    corpus_stats(train_df, 'body_training', 'training'),
])
stats_df


,dataset,Number of records,Total words,Unique words (types)
0,display,13214,1180453,31269
1,training,13214,1138740,27441
